# Lab 1. Wine quality regression

Predict the expert grade `quality` (0–10 scale) of a wine from 11 laboratory measurements.

- **Data:** [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality), red and white wines merged, 6497 rows.
- **Metric:** RMSE, required level **< 0.70**; MAE as a secondary metric.
- **Plan:** data and EDA → honest split → baseline → Linear Regression, Random Forest, Gradient Boosting → comparison → feature importance → error analysis.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import KFold, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message="The least populated class")
warnings.filterwarnings("ignore", message="The groups parameter is ignored")
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "quality"
PALETTE = {"red": "#b2182b", "white": "#d8b365"}

## 1. Data

In [ ]:
red = pd.read_csv("data/winequality-red.csv", sep=";").assign(wine_type="red")
white = pd.read_csv("data/winequality-white.csv", sep=";").assign(wine_type="white")
wine = pd.concat([red, white], ignore_index=True)
wine.columns = wine.columns.str.replace(" ", "_")
wine["is_red"] = (wine["wine_type"] == "red").astype(int)
FEATURES = [c for c in wine.columns if c not in (TARGET, "wine_type")]
CHEMISTRY = [c for c in FEATURES if c != "is_red"]

print(f"wines: {len(wine)} (red {len(red)}, white {len(white)}), features: {len(FEATURES)}")
print(f"missing values: {wine.isna().sum().sum()}, duplicate rows: {wine.duplicated().sum()}")
wine[[*FEATURES, TARGET]].describe().T[["mean", "std", "min", "50%", "max"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.countplot(data=wine, x=TARGET, hue="wine_type", palette=PALETTE, ax=ax)
ax.set_title("Quality distribution")
plt.show()
wine[TARGET].value_counts(normalize=True).sort_index().round(3).to_frame("share").T

**Findings**

- All features are numeric, no missing values — nothing to impute. The wine type is encoded as `is_red`.
- Scales differ by orders of magnitude (std of `density` is 0.003, of `total_sulfur_dioxide` — 56): linear coefficients will need standardization to be comparable; trees do not care.
- **1177 rows (18%) are exact duplicates.** If copies of one wine get into both train and test, the model just remembers the answer — the split must keep them together.
- **Long right tails / outliers:** `free_sulfur_dioxide` up to 289 (median 29), `residual_sugar` up to 65.8 (median 3), `density` up to 1.039. They look like real (very sweet or heavily sulphured) wines rather than errors, so they are kept. Trees are not affected by them; for linear regression they are influential points.
- The target is **imbalanced**: grades 5–6 cover 77% of wines, grades 3 and 9 — only 35 wines. The models will see few extreme examples and will likely be weak on them.

## 2. Train/test split

- **80/20** (5197 / 1300): enough data to train, and a test large enough for a stable estimate.
- **`StratifiedGroupKFold`**, first fold = test. *Stratified* by `quality`, so rare grades are in both parts in the same share. *Grouped* by identical rows, so duplicates never end up on both sides.
- `random_state=42` for reproducibility.
- Grade 9 has only 5 wines, so it cannot appear in every one of 5 folds; scikit-learn warns about this, and the warning is silenced in the first cell. The other grades are stratified normally.
- The test is used **once**, at the end. All choices (hyperparameters, best model) are made by 5-fold CV on the training part.

In [ ]:
X, y = wine[FEATURES], wine[TARGET]
groups = wine.groupby(FEATURES, sort=False).ngroup()
CV = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(CV.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
train = wine.iloc[train_idx]


def cv_rmse(model: object, cv: object = CV) -> tuple[float, float]:
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
    )
    return -scores["train_score"].mean(), -scores["test_score"].mean()


shared = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"train: {len(train_idx)}, test: {len(test_idx)}, duplicates shared by both: {len(shared)}")

probe = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED)
print(
    f"CV RMSE with random KFold: {cv_rmse(probe, KFold(5, shuffle=True, random_state=SEED))[1]:.3f}"
)
print(f"CV RMSE with grouped CV:   {cv_rmse(probe)[1]:.3f}")

pd.DataFrame(
    {"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}
).sort_index().round(3).T

**Findings**

- Grade shares in train and test match, and no duplicate is shared between them.
- The same forest gets **0.61** with a random split of the training part and **0.70** with the grouped one. The only difference between the two runs is whether copies of a wine may be split between training and validation folds, so most of that 0.08 is remembered duplicates, not quality — the "unusually good result" the task warns about.
- **Why not evaluate many times on one test set:** if models and parameters are chosen by the test score, we gradually fit the noise of those particular 1300 wines. The estimate becomes optimistic, and the test turns into a validation set. That is why every decision here is made by CV on train.

## 3. Exploratory analysis

Done on the **training part only**, so that no decision is based on the test answers.

In [ ]:
corr = train[[*FEATURES, TARGET]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 7}, ax=ax
)
ax.set_title("Pearson correlation (train)")
plt.show()

pd.DataFrame(
    {
        "all": corr[TARGET].drop([TARGET, "is_red"]),
        "red": train[train["is_red"] == 1][CHEMISTRY].corrwith(train[train["is_red"] == 1][TARGET]),
        "white": train[train["is_red"] == 0][CHEMISTRY].corrwith(
            train[train["is_red"] == 0][TARGET]
        ),
    }
).sort_values("all", key=abs, ascending=False).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, feature in zip(axes, ["alcohol", "volatile_acidity", "free_sulfur_dioxide"], strict=True):
    sns.boxplot(
        data=train,
        x=TARGET,
        y=feature,
        hue="wine_type",
        palette=PALETTE,
        fliersize=1,
        legend=ax is axes[0],
        ax=ax,
    )
axes[2].set_ylim(0, 120)
fig.tight_layout()
plt.show()

**Findings**

- **`alcohol` is the strongest signal** (r = 0.44). Riper grapes have more sugar, which ferments into more alcohol and a fuller taste. But r² ≈ 0.19: alcohol explains less than 20% of the variance, so the grade also depends on things that are not measured, and RMSE near zero is unrealistic.
- The alcohol effect is **non-linear**: it is flat for grades 3–5 and grows only from grade 5 upwards. A straight line cannot describe it, so trees should beat linear regression.
- **`density` is alcohol in disguise** (r = −0.68 with alcohol, 0.55 with sugar). It adds little new information and will distort the linear coefficients.
- **`volatile_acidity`** (vinegar defect) marks bad wines (r = −0.27, for red −0.39). For white wines it stays around 0.25–0.28 at every grade except 4, for red it keeps falling up to grade 7: high values mean "bad", but low values do not guarantee "good".
- **Red and white wines behave differently:** `sulphates` correlates with quality for red (0.25) but not for white, and `citric_acid` behaves the same way. That is why `is_red` stays in the features. Interactions like this are easy for trees and impossible for a plain linear model.
- `free_sulfur_dioxide` has almost no linear correlation, yet in white wines a very low level (≤ 10) comes with low grades. The effect is non-linear.
- Boxes of neighboring grades overlap heavily, so an error of about half a grade is normal even for a good model.

## 4. Baseline and metrics

**Baseline** is one constant for every wine. RMSE is minimized by the **mean** ($\frac{d}{dc}\sum (y_i - c)^2 = 0 \Rightarrow c = \bar y$), MAE by the median. The main metric is RMSE, so the baseline predicts the **mean of the training grades**.

$$\mathrm{MAE} = \frac{1}{n}\sum |y_i - \hat y_i| \qquad \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat y_i)^2}$$

- **MAE** is the average size of the error in grade points ("off by half a grade on average").
- **RMSE** is the typical error, with large errors weighted more. It matches the MSE loss the models are trained on.
- **RMSE reacts more to large errors** because errors are squared: an error of 2 counts 4 times more than an error of 1, and one error of 3 equals nine errors of 1.
- **MAE ≤ RMSE always.** They are equal only when all errors are the same size. The gap grows when there are a few big mistakes among many small ones; here those are the rare grades.

## 5. Models

- **Linear Regression**: `StandardScaler` + `LinearRegression` in one pipeline, so the scaler is fitted on training data only. Scaling does not change the predictions, but it makes the coefficients comparable: each one is the effect of +1 standard deviation of its feature.
- **Random Forest**: the average of 500 deep trees, each trained on a random subsample of rows; with `max_features` < 1 each split also chooses among a random subset of features. Averaging cancels out the mistakes of individual trees.
- **Gradient Boosting**: 500 shallow trees built one after another, each correcting the errors of the previous ones with a small step (learning rate).

For each tree model, the default settings and one adjusted variant are compared by CV on train. The adjusted variants are standard textbook settings fixed in advance, not tuned on the test: `max_features="sqrt"` is the classic random-forest recommendation (more diverse trees), and for boosting a smaller learning rate with more trees, row subsampling and a minimum leaf size is the usual regularized recipe. Systematic tuning is the topic of lab 3.

In [ ]:
candidates = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "RF default": RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=SEED),
    "RF max_features=sqrt": RandomForestRegressor(
        n_estimators=500, max_features="sqrt", n_jobs=-1, random_state=SEED
    ),
    "GB default": GradientBoostingRegressor(random_state=SEED),
    "GB 500 trees, lr=0.03, depth=5": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        min_samples_leaf=5,
        random_state=SEED,
    ),
}
cv_table = pd.DataFrame(
    [cv_rmse(model) for model in candidates.values()],
    index=list(candidates),
    columns=["train RMSE", "CV RMSE"],
)
cv_table

- **RF with `max_features=sqrt`** (0.685) and **tuned boosting** (0.697) win their families on CV and are the ones evaluated on the test set.
- Linear regression has train ≈ CV (0.733 vs 0.736): it does not overfit, it **underfits** — the model is too simple for this data.
- Trees have train RMSE well below CV (RF 0.22 vs 0.69, GB 0.43–0.62 vs 0.70). For a forest this is normal (single trees memorize, averaging generalizes); only the CV score matters.

In [ ]:
def best_of(prefix: str) -> str:
    return cv_table.loc[cv_table.index.str.startswith(prefix), "CV RMSE"].idxmin()


models = {
    "Baseline": candidates["Baseline (mean)"],
    "Linear Regression": candidates["Linear Regression"],
    "Random Forest": candidates[best_of("RF")],
    "Gradient Boosting": candidates[best_of("GB")],
}
rows = []
for name, model in models.items():
    prediction = model.fit(X_train, y_train).predict(X_test)
    rows.append(
        {
            "model": name,
            "RMSE": root_mean_squared_error(y_test, prediction),
            "MAE": mean_absolute_error(y_test, prediction),
        }
    )
results = pd.DataFrame(rows).set_index("model")
results["beats baseline"] = results["RMSE"] < results.loc["Baseline", "RMSE"]
results["RMSE < 0.70"] = results["RMSE"] < 0.70
results

## 6. Comparison

All models use the same train, the same test and the same CV folds.

- **Every model beats the baseline** (0.873, which is roughly the spread of the grades). The baseline predicts the mean because it is optimal for RMSE, the main metric; a median constant would give a slightly better MAE (0.637 instead of 0.686).
- **Linear Regression (0.731) does not reach 0.70.** The relations are non-linear (alcohol, volatile acidity thresholds, free SO₂) and depend on the wine type, and a weighted sum of features cannot express that.
- **Random Forest (0.670) and Gradient Boosting (0.685) pass.** Both fit the training data far better than new data (train 0.22 and 0.43), so the train error says little here; the CV score decides. The forest is slightly better, probably because the grade is noisy (a subjective integer score, neighboring grades overlap). The forest averages 500 *independent* deep trees, so their individual mistakes cancel out. Boosting builds trees *one after another* on the remaining errors, and when those errors are mostly noise, extra trees add little: more capacity lowered its train RMSE from 0.62 to 0.43 but CV only from 0.701 to 0.697. The gap between RF and GB (0.015) is small compared with the gap to LR (0.06).
- **Ceiling:** even the best model removes only about 41% of the squared error of the constant baseline ($1 - (0.670/0.873)^2$). The rest depends on grape variety, aroma and the taste of the experts, none of which is in the data.

## 7. Feature importance

In [ ]:
linear, forest, boosting = (
    models[n] for n in ["Linear Regression", "Random Forest", "Gradient Boosting"]
)
importance = pd.DataFrame(
    {
        "LR raw coef": LinearRegression().fit(X_train, y_train).coef_,
        "LR std coef": linear[-1].coef_,
        "RF": forest.feature_importances_,
        "GB": boosting.feature_importances_,
        "corr": corr[TARGET].drop(TARGET),
    },
    index=FEATURES,
).sort_values("RF", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
order = importance.index[::-1]
for ax, column, color in zip(
    axes, ["LR std coef", "RF", "GB"], ["#4393c3", "#5aae61", "#8073ac"], strict=True
):
    importance.loc[order, column].plot.barh(ax=ax, color=color)
    ax.set_title(column)
fig.tight_layout()
plt.show()
importance

**Findings**

- **Raw linear coefficients are misleading.** `density` gets −101 only because its std is 0.003. After standardization the largest are density (−0.31), residual sugar (+0.30), alcohol (+0.27) and volatile acidity (−0.24). Density and sugar are inflated by **multicollinearity**: density ≈ f(alcohol, sugar), so the two coefficients compensate each other, even though sugar alone does not correlate with quality.
- **RF and GB agree on the top:** `alcohol` is first by far, `volatile_acidity` is second, and `free_sulfur_dioxide` is in the top 4.
- **Difference:** `density` is 3rd in RF but low in GB. In a forest, alcohol is often missing from the random feature subset, so the tree splits on its substitute, density. Boosting uses alcohol directly, and density then adds nothing.
- **`is_red` is almost unused (≈ 0.003)** although EDA showed that red and white wines differ. This is not a contradiction: the type is easy to recognize from its chemistry (in section 3 red and white wines differ 2–3 times in total SO₂, volatile acidity and chlorides), so the trees split on those features and never need the flag. Impurity importance gives the credit to the feature that was actually used.
- **Against EDA:** alcohol and volatile acidity are confirmed. `free_sulfur_dioxide` has almost zero correlation but high tree importance, because its effect is non-linear (bad for white wines when it is too low). Correlation misses this, trees catch it. Density and chlorides correlate with quality, but mostly through alcohol and wine type: boosting ranks them low, while in the forest they stay 3rd and 5th as substitutes when alcohol or the type features are not in the random subset.

**Does feature importance mean that changing a feature changes the quality? No.**
Importance shows how much the model *uses* a feature to predict in this data. That is an association, not a cause. Alcohol is probably a marker of ripe grapes and a good vintage; adding spirit to a wine will not make it better. Correlated features also swap importance with each other (alcohol ↔ density). Proving a cause needs an experiment, for example a blind tasting of one wine at different alcohol levels.

## 8. Error analysis (Random Forest)

In [ ]:
errors = wine.iloc[test_idx].assign(prediction=forest.predict(X_test))
errors["error"] = errors[TARGET] - errors["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
sns.histplot(errors["error"], bins=40, color="#4393c3", ax=axes[0])
axes[0].set_title("Error = true - predicted")
sns.boxplot(data=errors, x=TARGET, y="prediction", color="#92c5de", fliersize=2, ax=axes[1])
axes[1].plot(range(7), range(3, 10), "r--", label="ideal")
axes[1].legend()
axes[1].set_title("Prediction by true quality")
fig.tight_layout()
plt.show()

within = errors["error"].abs()
print(f"mean error: {errors['error'].mean():.3f}, median |error|: {within.median():.2f}")
print(f"|error| < 0.5: {(within < 0.5).mean():.0%}, |error| < 1: {(within < 1).mean():.0%}")
print(f"predictions range: {errors['prediction'].min():.2f} - {errors['prediction'].max():.2f}")
errors.groupby(TARGET).agg(
    count=("error", "size"),
    mean_prediction=("prediction", "mean"),
    MAE=("error", lambda e: e.abs().mean()),
)

In [ ]:
errors.reindex(errors["error"].abs().nlargest(5).index)[
    [
        "wine_type",
        TARGET,
        "prediction",
        "alcohol",
        "volatile_acidity",
        "free_sulfur_dioxide",
        "sulphates",
    ]
]

**Findings**

- Errors are centered at zero (mean −0.004) and roughly symmetric (see the histogram). 58% of wines are predicted within ±0.5 of the true grade, 88% within ±1.
- **Predictions are squeezed towards the middle:** true grades range over 3–9, predictions only over 4.58–7.31. The model overestimates bad wines and underestimates good ones. For grades 5–6 the MAE is 0.35–0.47; for grades 3 and 9 it is about 2.3.
- **Why rare grades are worse:**
  1. *Few examples:* train has 24 wines of grade 3 and 4 of grade 9, so even in the "bad" or "good" regions the leaves are dominated by grades 5–6.
  2. *The loss pulls to the mean:* minimizing MSE predicts the average grade of similar wines, and with overlapping grades that average is close to 5.8.
  3. *Averaging:* the forest averages 500 trees, which smooths out extreme predictions even more.
- **Worst cases:**
  - #5008: true 3, predicted 6.0. Its measurements are those of a typical grade-6 wine (alcohol 11, volatile acidity 0.23), so the defect is in something not measured (aroma, cork) or in the grading itself.
  - #2815 / #2818: the same wine twice, true 8, predicted 5.2. High alcohol (11.7) points to a good wine, but volatile acidity 0.37 is far above the typical 0.25 of good white wines and is a marker of low grades. The model trusted the defect marker, the experts did not.
  - #1852: true 3, predicted 5.5. Free SO₂ is only 5, a level at which white wines are graded low, so the model lowered its prediction. The high alcohol (11.7) pulled it back up, and it stopped far from 3.

## Conclusions

| Model | CV RMSE | Test RMSE | Test MAE |
| --- | --- | --- | --- |
| Baseline | 0.873 | 0.873 | 0.686 |
| Linear Regression | 0.736 | 0.731 | 0.563 |
| **Random Forest** | **0.685** | **0.670** | **0.516** |
| Gradient Boosting | 0.697 | 0.685 | 0.525 |

- **Final model: Random Forest, test RMSE 0.670 < 0.70.** It was chosen by CV on train; the test was used once.
- Duplicates (18% of the data) would inflate the result to about 0.61 with a random split. The grouped split keeps the estimate honest.
- Linear regression underfits because the relations are non-linear and depend on the wine type. Tree ensembles capture this.
- Key features: `alcohol` and `volatile_acidity`, plus `free_sulfur_dioxide` through a non-linear effect. Feature importance shows association, not cause.
- The model is accurate on typical wines (grades 5–6) and pulls rare grades towards the middle.